# PC5 v9 - Multiclass Leaf + Disease YOLO-Seg
Train one YOLO segmentation model with two classes: `leaf` and `disease_region`. Inputs are RGB leaf crops, not black-background isolated leaves. Holdout is excluded from train/val/test and used only for real unseen qualitative predictions.

## Setup
Install/check Ultralytics and configure deterministic GPU execution.

In [ ]:
import json, math, os, random, re, shutil, subprocess, sys, time, traceback, zipfile
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image

try:
    from ultralytics import YOLO
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print({"python": sys.version.split()[0], "torch": torch.__version__, "cuda_available": torch.cuda.is_available(), "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})


## Paths And Outputs
Create Kaggle working folders, QA gates, classes, and run summary.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
CATEGORIES = ["healthy_leaves","small_diseased_regions","large_diseased_regions","simple_backgrounds","complex_backgrounds","multiple_leaves","partial_occlusions"]
MIN_ACCEPTED_DISEASE_POSITIVES = 500
MIN_HEALTHY_NEGATIVES = 1000

PREPARED_DIR = WORKING_DIR / "prepared_multiclass_yolo_dataset"
PREVIEW_DIR = WORKING_DIR / "multiclass_pseudo_labels_preview"
REAL_HOLDOUT_DIR = WORKING_DIR / "real_holdout_predictions"
HOLDOUT_DIAG_DIR = WORKING_DIR / "holdout_diagnostic_reference"
QUAL_TEST_DIR = WORKING_DIR / "qualitative_multiclass_results"
QUANT_DIR = WORKING_DIR / "quantitative_metrics"
RUNS_DIR = WORKING_DIR / "runs_multiclass"
SUMMARY_PATH = WORKING_DIR / "run_summary.json"

for p in [
    PREPARED_DIR,
    PREVIEW_DIR / "accepted",
    PREVIEW_DIR / "rejected",
    PREVIEW_DIR / "healthy_leaf_only",
    REAL_HOLDOUT_DIR,
    HOLDOUT_DIAG_DIR,
    QUAL_TEST_DIR,
    QUANT_DIR,
    RUNS_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)
for c in CATEGORIES:
    (REAL_HOLDOUT_DIR / "by_category" / c).mkdir(parents=True, exist_ok=True)
    (HOLDOUT_DIAG_DIR / "by_category" / c).mkdir(parents=True, exist_ok=True)

progress = {
    "status": "running",
    "stage": "start",
    "task": "multiclass leaf + disease_region segmentation",
    "pseudo_label_source": "leaf heuristic v7 + conservative disease heuristic; no YOLO/SAM pseudo-label source",
    "holdout_policy": "67 holdout images excluded from train/val/test; real qualitative predictions only",
    "classes": {"0": "leaf", "1": "disease_region"},
    "qa_gate": {
        "min_accepted_disease_positives": MIN_ACCEPTED_DISEASE_POSITIVES,
        "min_healthy_negatives": MIN_HEALTHY_NEGATIVES,
    },
    "artifacts": [],
}

def save_progress():
    SUMMARY_PATH.write_text(json.dumps(progress, indent=2), encoding="utf-8")

save_progress()


## Leaf Helpers
Reuse the v7 leaf tissue + shape heuristic, dataset discovery, polygon conversion, overlays, metrics, and zipping.

In [ ]:
def read_rgb(path):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f"Could not read image: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def write_rgb(path, rgb, quality=92):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(rgb).save(path, quality=quality)

def normalize_image_id(name):
    stem = Path(str(name)).stem.lower()
    stem = re.sub(r"[^a-z0-9]+", "_", stem)
    return re.sub(r"_+", "_", stem).strip("_")

def extract_holdout_original(filename):
    parts = filename.split("__", 2)
    if len(parts) != 3 or len(parts[0]) != 3 or not parts[0].isdigit():
        return None
    return parts[2]

def find_plantvillage_root(input_root):
    candidates = [
        input_root / "plantdisease" / "PlantVillage",
        input_root / "datasets" / "emmarex" / "plantdisease" / "PlantVillage",
    ]
    for c in candidates:
        if c.exists():
            return c
    for c in input_root.glob("**/PlantVillage"):
        if c.is_dir():
            return c
    raise FileNotFoundError("PlantVillage root not found")

def find_holdout_root(input_root, categories):
    candidates = [
        input_root / "cv-pc5-v3-plantvillage-segmentation-holdout",
        input_root / "datasets" / "jeffreyamc" / "cv-pc5-v3-plantvillage-segmentation-holdout",
    ]
    for c in candidates:
        if c.exists() and all((c / x).exists() for x in categories):
            return c
    for c in input_root.glob("**"):
        if c.is_dir() and all((c / x).exists() for x in categories):
            return c
    raise FileNotFoundError("Holdout root not found")

def split_stratified(df, group_col="source_class", train_ratio=0.70, val_ratio=0.15):
    chunks = []
    for _, g in df.groupby(group_col, sort=True):
        g = g.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        n = len(g)
        n_train = int(round(n * train_ratio))
        n_val = int(round(n * val_ratio))
        if n >= 3:
            n_train = max(1, min(n_train, n - 2))
            n_val = max(1, min(n_val, n - n_train - 1))
        else:
            n_train, n_val = max(1, n - 1), 0
        g.loc[: n_train - 1, "split"] = "train"
        if n_val:
            g.loc[n_train : n_train + n_val - 1, "split"] = "val"
        g.loc[n_train + n_val :, "split"] = "test"
        chunks.append(g)
    return pd.concat(chunks, ignore_index=True)

def fill_holes(mask):
    mask = (mask > 0).astype(np.uint8) * 255
    h, w = mask.shape
    flood = mask.copy()
    cv2.floodFill(flood, np.zeros((h + 2, w + 2), np.uint8), (0, 0), 255)
    holes = cv2.bitwise_not(flood)
    return cv2.bitwise_or(mask, holes)

def reconstruct_from_seed(seed, candidate, iterations=48):
    seed = (seed > 0).astype(np.uint8) * 255
    candidate = (candidate > 0).astype(np.uint8) * 255
    kernel = np.ones((5, 5), np.uint8)
    cur = cv2.bitwise_and(seed, candidate)
    for _ in range(iterations):
        nxt = cv2.bitwise_and(cv2.dilate(cur, kernel, iterations=1), candidate)
        if np.array_equal(nxt, cur):
            break
        cur = nxt
    return cur

def keep_reasonable_components(mask, max_components=3, min_area_ratio=0.004):
    h, w = mask.shape
    n, labels, stats, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), 8)
    if n <= 1:
        return np.zeros_like(mask)
    min_area = max(64, int(min_area_ratio * h * w))
    order = np.argsort(stats[1:, cv2.CC_STAT_AREA])[::-1] + 1
    clean = np.zeros_like(mask)
    for lab_id in order[:max_components]:
        if stats[lab_id, cv2.CC_STAT_AREA] >= min_area:
            clean[labels == lab_id] = 255
    if clean.sum() == 0:
        clean[labels == int(order[0])] = 255
    return clean

def leaf_tissue_shape_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    h, s, v = cv2.split(hsv)
    l, a, b = cv2.split(lab)
    r = rgb[:, :, 0].astype(np.int16)
    g = rgb[:, :, 1].astype(np.int16)
    bb = rgb[:, :, 2].astype(np.int16)
    exg = cv2.normalize((2 * g - r - bb).astype(np.float32), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    exg_otsu = cv2.threshold(exg, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1] > 0
    sat_otsu = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1] > 0
    chroma = cv2.threshold(cv2.absdiff(a, b), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1] > 0

    green = (h >= 22) & (h <= 100) & (s >= 18) & (v >= 25)
    yellow = (h >= 12) & (h <= 48) & (s >= 18) & (v >= 45)
    brown = (((h <= 24) | (h >= 165)) & (s >= 18) & (v >= 22) & (v <= 220))
    dark_leaf_like = (v <= 105) & (s >= 15) & (gray >= 12)

    # Keep the v4-style foreground as the anchor. Broad chroma/Otsu masks are
    # useful only as seed evidence; if used as candidates they can absorb gray
    # textured backgrounds and train YOLO on bad pseudo-labels.
    seed_bool = (green | yellow | (exg_otsu & (s >= 18)) | (chroma & sat_otsu & (s >= 20))) & (v >= 20)
    seed = seed_bool.astype(np.uint8) * 255
    seed_neighborhood = cv2.dilate(seed, np.ones((21, 21), np.uint8), iterations=2) > 0
    candidate_bool = green | yellow | brown | (dark_leaf_like & seed_neighborhood)
    candidate = (candidate_bool & (v >= 12)).astype(np.uint8) * 255

    kernel3 = np.ones((3, 3), np.uint8)
    kernel5 = np.ones((5, 5), np.uint8)
    seed = cv2.morphologyEx(seed, cv2.MORPH_OPEN, kernel3, iterations=1)
    candidate = cv2.morphologyEx(candidate, cv2.MORPH_CLOSE, kernel5, iterations=2)
    candidate = cv2.morphologyEx(candidate, cv2.MORPH_OPEN, kernel3, iterations=1)
    seed = keep_reasonable_components(seed, max_components=4, min_area_ratio=0.002)
    mask = reconstruct_from_seed(seed, candidate, iterations=64)
    seed_fallback = cv2.morphologyEx(seed, cv2.MORPH_CLOSE, kernel5, iterations=3)
    seed_fallback = fill_holes(keep_reasonable_components(seed_fallback, max_components=3, min_area_ratio=0.004))
    if mask.sum() == 0:
        mask = seed_fallback
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel5, iterations=3)
    before_fill_area = int((mask > 0).sum())
    mask = fill_holes(mask)
    mask = keep_reasonable_components(mask, max_components=3, min_area_ratio=0.004)
    mask = cv2.medianBlur(mask, 5)
    # If reconstruction still leaked into the background, fall back to the
    # conservative anchor mask rather than publishing a false full-image leaf.
    simple_fg = mask > 0
    border = np.zeros_like(simple_fg, dtype=bool)
    border[0, :] = border[-1, :] = border[:, 0] = border[:, -1] = True
    area_ratio = float(simple_fg.mean())
    border_touch = float(np.logical_and(simple_fg, border).sum() / max(1, border.sum()))
    if area_ratio > 0.88 or (area_ratio > 0.60 and border_touch > 0.55):
        mask = cv2.medianBlur(seed_fallback, 5)
        before_fill_area = int((mask > 0).sum())
        mask = fill_holes(mask)
        mask = keep_reasonable_components(mask, max_components=3, min_area_ratio=0.004)
        mask = cv2.medianBlur(mask, 5)
    after_fill_area = int((mask > 0).sum())
    return mask, before_fill_area, after_fill_area

def mask_quality(mask, before_fill_area=None, after_fill_area=None):
    h, w = mask.shape
    fg = mask > 0
    area = int(fg.sum())
    area_ratio = float(area / max(1, h * w))
    n, labels, stats, _ = cv2.connectedComponentsWithStats(fg.astype(np.uint8), 8)
    comp_count = max(0, n - 1)
    border = np.zeros_like(fg, dtype=bool)
    border[0, :] = border[-1, :] = border[:, 0] = border[:, -1] = True
    border_touch_ratio = float(np.logical_and(fg, border).sum() / max(1, border.sum()))
    if area:
        ys, xs = np.where(fg)
        x1, x2, y1, y2 = xs.min(), xs.max(), ys.min(), ys.max()
        bbox_area = int((x2 - x1 + 1) * (y2 - y1 + 1))
        bbox_fill_ratio = float(area / max(1, bbox_area))
        bbox_image_ratio = float(bbox_area / max(1, h * w))
    else:
        bbox_fill_ratio = 0.0
        bbox_image_ratio = 0.0
    if before_fill_area is None or after_fill_area is None:
        hole_fill_ratio = 0.0
    else:
        hole_fill_ratio = float(max(0, after_fill_area - before_fill_area) / max(1, after_fill_area))
    reasons = []
    if area == 0:
        reasons.append("empty_mask")
    if area_ratio < 0.025:
        reasons.append("small_mask_area")
    if area_ratio > 0.88:
        reasons.append("large_mask_area")
    if comp_count > 4:
        reasons.append("many_components")
    if border_touch_ratio > 0.80 and area_ratio > 0.60:
        reasons.append("excessive_border_touch")
    if border_touch_ratio > 0.55 and bbox_fill_ratio > 0.90 and area_ratio > 0.55:
        reasons.append("border_touch_background_like")
    if bbox_fill_ratio > 0.97 and bbox_image_ratio > 0.60:
        reasons.append("background_like_solid_region")
    if bbox_fill_ratio < 0.18 and area_ratio > 0.03:
        reasons.append("fragmented_sparse_shape")
    return {
        "mask_area_ratio": area_ratio,
        "component_count": int(comp_count),
        "border_touch_ratio": border_touch_ratio,
        "hole_fill_ratio": hole_fill_ratio,
        "bbox_fill_ratio": bbox_fill_ratio,
        "bbox_image_ratio": bbox_image_ratio,
        "weak_reasons": ";".join(reasons),
        "accepted": len(reasons) == 0,
    }

def masks_to_yolo_lines(mask, class_id=0, min_area_ratio=0.0015, epsilon_ratio=0.0025, max_contours=8):
    h, w = mask.shape
    contours, _ = cv2.findContours((mask > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    lines, min_area = [], max(16, h * w * min_area_ratio)
    for cnt in contours[:max_contours]:
        if cv2.contourArea(cnt) < min_area:
            continue
        eps = max(1.0, epsilon_ratio * cv2.arcLength(cnt, True))
        approx = cv2.approxPolyDP(cnt, eps, True).reshape(-1, 2)
        if len(approx) < 3:
            continue
        coords = []
        for x, y in approx:
            coords.extend([float(np.clip(x / w, 0, 1)), float(np.clip(y / h, 0, 1))])
        lines.append(str(class_id) + " " + " ".join(f"{v:.6f}" for v in coords))
    return lines

def label_file_to_mask(label_path, shape):
    h, w = shape
    mask = np.zeros((h, w), dtype=np.uint8)
    p = Path(label_path)
    if not p.exists():
        return mask
    for line in p.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        vals = [float(v) for v in parts[1:]]
        pts = [[int(round(x * w)), int(round(y * h))] for x, y in zip(vals[0::2], vals[1::2])]
        if len(pts) >= 3:
            cv2.fillPoly(mask, [np.array(pts, dtype=np.int32)], 255)
    return mask

def result_to_mask(result, shape):
    h, w = shape
    mask = np.zeros((h, w), dtype=np.uint8)
    masks = getattr(result, "masks", None)
    if masks is None:
        return mask
    data = getattr(masks, "data", None)
    if data is not None and len(data):
        for arr in data.detach().cpu().numpy():
            m = (arr > 0.5).astype(np.uint8) * 255
            if m.shape != (h, w):
                m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
            mask = cv2.bitwise_or(mask, m)
    elif getattr(masks, "xy", None) is not None:
        for poly in masks.xy:
            if poly is not None and len(poly) >= 3:
                cv2.fillPoly(mask, [np.array(poly, dtype=np.int32)], 255)
    return mask

def mask_iou(a, b):
    aa, bb = a > 0, b > 0
    union = np.logical_or(aa, bb).sum()
    return 1.0 if union == 0 else float(np.logical_and(aa, bb).sum() / union)

def overlay_mask(rgb, mask, color=(40, 180, 80), alpha=0.45):
    out = rgb.copy()
    c = np.array(color, dtype=np.uint8)
    out[mask > 0] = (out[mask > 0] * (1 - alpha) + c * alpha).astype(np.uint8)
    edges = cv2.Canny((mask > 0).astype(np.uint8) * 255, 50, 150)
    out[edges > 0] = np.array([255, 40, 40], dtype=np.uint8)
    return out

def side_by_side(images):
    h = min(img.shape[0] for img in images)
    resized = []
    for img in images:
        if img.shape[0] != h:
            w = int(img.shape[1] * h / img.shape[0])
            img = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)
        resized.append(img)
    return np.concatenate(resized, axis=1)

def zip_dir(folder, zip_path):
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=folder)
    return zip_path

def metrics_from_results(metrics):
    values = {}
    for attr in ["box", "seg"]:
        obj = getattr(metrics, attr, None)
        if obj is None:
            continue
        for key in ["mp", "mr", "map50", "map", "map75"]:
            val = getattr(obj, key, None)
            if val is not None:
                try:
                    values[f"pseudo_{attr}_{key}"] = float(val)
                except Exception:
                    pass
    return values

def assert_only_class_zero(label_root):
    bad = []
    for p in Path(label_root).glob("**/*.txt"):
        for line in p.read_text(encoding="utf-8").splitlines():
            if line.strip() and line.split()[0] != "0":
                bad.append(str(p))
                break
    if bad:
        raise AssertionError(f"Non-zero class ids in labels: {bad[:5]}")


## Discover Inputs And Exclude Holdout
Build holdout manifest and remove matching original PlantVillage files before any split.

In [ ]:
progress["stage"] = "discover_inputs"; save_progress()
plant_root = find_plantvillage_root(INPUT_ROOT)
holdout_root = find_holdout_root(INPUT_ROOT, CATEGORIES)
holdout_records, holdout_norm_ids = [], set()
for category in CATEGORIES:
    for p in sorted((holdout_root / category).glob("*.jpg")):
        original = extract_holdout_original(p.name)
        if original is None:
            raise ValueError(f"Bad holdout filename: {p.name}")
        norm = normalize_image_id(original)
        holdout_norm_ids.add(norm)
        holdout_records.append({"category": category, "holdout_filename": p.name, "original_filename": original, "normalized_original_id": norm, "path": str(p)})
holdout_df = pd.DataFrame(holdout_records)
if len(holdout_df) != 67:
    raise AssertionError(f"Expected 67 holdout images, found {len(holdout_df)}")
holdout_manifest_path = WORKING_DIR / "holdout_manifest.csv"
holdout_df.to_csv(holdout_manifest_path, index=False)

records, excluded = [], []
for class_dir in sorted([p for p in plant_root.iterdir() if p.is_dir()]):
    is_healthy = "healthy" in class_dir.name.lower()
    paths = []
    for ext in ["*.jpg", "*.JPG", "*.jpeg", "*.png"]:
        paths.extend(class_dir.glob(ext))
    for p in sorted(paths):
        norm = normalize_image_id(p.name)
        rec = {"source_class": class_dir.name, "filename": p.name, "normalized_id": norm, "is_healthy": is_healthy, "path": str(p)}
        (excluded if norm in holdout_norm_ids else records).append(rec)
source_df = pd.DataFrame(records)
if set(source_df["normalized_id"]) & holdout_norm_ids:
    raise AssertionError("Holdout leaked into candidate source set")
progress.update({
    "plant_root": str(plant_root),
    "holdout_root": str(holdout_root),
    "holdout_count": int(len(holdout_df)),
    "excluded_from_plantvillage": int(len(excluded)),
    "source_images_after_holdout_exclusion": int(len(source_df)),
})
progress["artifacts"].append(str(holdout_manifest_path))
save_progress()
print({k: progress[k] for k in ["holdout_count", "excluded_from_plantvillage", "source_images_after_holdout_exclusion"]})


## Multiclass Helpers
Create RGB crops, conservative disease masks, multiclass labels, prediction masks, and label validation.

In [ ]:
FOCAL_DISEASE_HINTS = [
    "bacterial_spot",
    "early_blight",
    "late_blight",
    "septoria",
    "target_spot",
    "leaf_mold",
    "spider_mites",
]
DIFFUSE_DISEASE_HINTS = ["yellowleaf", "yellow_leaf", "curl_virus", "mosaic"]

def normalized_class_name(name):
    return normalize_image_id(str(name))

def disease_policy(source_class):
    n = normalized_class_name(source_class)
    if any(x in n for x in DIFFUSE_DISEASE_HINTS):
        return "exclude_diffuse"
    if any(x in n for x in FOCAL_DISEASE_HINTS):
        return "focal_positive"
    return "unknown_diseased_exclude"

def disease_area_max_for_class(source_class):
    n = normalized_class_name(source_class)
    if "late_blight" in n or "leaf_mold" in n:
        return 0.35
    return 0.20

def bbox_from_mask(mask, pad_ratio=0.10):
    h, w = mask.shape
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return (0, 0, w - 1, h - 1)
    x1, x2 = int(xs.min()), int(xs.max())
    y1, y2 = int(ys.min()), int(ys.max())
    pad = int(max(x2 - x1 + 1, y2 - y1 + 1) * pad_ratio)
    return (max(0, x1 - pad), max(0, y1 - pad), min(w - 1, x2 + pad), min(h - 1, y2 + pad))

def crop_rgb_mask(rgb, mask, bbox):
    x1, y1, x2, y2 = bbox
    return rgb[y1:y2 + 1, x1:x2 + 1].copy(), mask[y1:y2 + 1, x1:x2 + 1].copy()

def paste_crop_mask(crop_mask, full_shape, bbox):
    out = np.zeros(full_shape, dtype=np.uint8)
    x1, y1, x2, y2 = bbox
    out[y1:y2 + 1, x1:x2 + 1] = crop_mask[: y2 - y1 + 1, : x2 - x1 + 1]
    return out

def disease_mask_conservative(crop_rgb, crop_leaf_mask, source_class):
    inside = crop_leaf_mask > 0
    leaf_area = int(inside.sum())
    if leaf_area == 0:
        return np.zeros(crop_leaf_mask.shape, dtype=np.uint8), {
            "accepted": False,
            "weak_reasons": "empty_leaf_mask",
            "disease_area_ratio_leaf": 0.0,
            "component_count": 0,
            "class_area_max": disease_area_max_for_class(source_class),
        }

    hsv = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2HSV)
    lab = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2LAB)
    gray = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2GRAY)
    h, s, v = cv2.split(hsv)
    l, a, b = cv2.split(lab)
    lab_inside = lab[inside].astype(np.float32)
    med = np.median(lab_inside, axis=0)
    dist = np.linalg.norm(lab.astype(np.float32) - med.reshape(1, 1, 3), axis=2)
    dist_inside = dist[inside]
    local_l = cv2.GaussianBlur(l.astype(np.float32), (0, 0), 5)
    contrast = np.abs(l.astype(np.float32) - local_l)
    contrast_thr = max(7.0, float(np.percentile(contrast[inside], 92)))

    brown_necrotic = (((h <= 26) | (h >= 165)) & (s >= 28) & (v >= 18) & (v <= 215))
    dark_internal = ((l <= np.percentile(l[inside], 14)) & (s >= 14) & (v >= 14))
    lab_outlier_strong = (dist >= max(18.0, float(np.percentile(dist_inside, 94)))) & (s >= 12)
    contrast_seed = (contrast >= contrast_thr) & (s >= 18) & ((v <= np.percentile(v[inside], 55)) | brown_necrotic)
    seeds = inside & (brown_necrotic | dark_internal | lab_outlier_strong | contrast_seed)

    yellow = ((h >= 16) & (h <= 48) & (s >= 24) & (v >= 70))
    lab_outlier_moderate = dist >= max(10.0, float(np.percentile(dist_inside, 80)))
    seed_neighborhood = cv2.dilate(seeds.astype(np.uint8) * 255, np.ones((7, 7), np.uint8), iterations=2) > 0
    candidate = inside & (seeds | (yellow & seed_neighborhood & lab_outlier_moderate))

    k3 = np.ones((3, 3), np.uint8)
    candidate_u8 = (candidate.astype(np.uint8) * 255)
    candidate_u8 = cv2.morphologyEx(candidate_u8, cv2.MORPH_OPEN, k3, iterations=1)
    candidate_u8 = cv2.morphologyEx(candidate_u8, cv2.MORPH_CLOSE, k3, iterations=1)

    n, labels, stats, _ = cv2.connectedComponentsWithStats((candidate_u8 > 0).astype(np.uint8), 8)
    clean = np.zeros(candidate_u8.shape, dtype=np.uint8)
    min_area = max(10, int(0.0005 * leaf_area))
    max_area = disease_area_max_for_class(source_class) * leaf_area
    leaf_edge = cv2.Canny((crop_leaf_mask > 0).astype(np.uint8) * 255, 50, 150) > 0
    leaf_edge_dil = cv2.dilate(leaf_edge.astype(np.uint8), np.ones((5, 5), np.uint8), iterations=1) > 0
    component_rejections = Counter()

    for lab_id in range(1, n):
        comp = labels == lab_id
        area = int(comp.sum())
        if area < min_area:
            component_rejections["component_too_small"] += 1
            continue
        if area > max_area:
            component_rejections["component_too_large"] += 1
            continue
        seed_ratio = float(seeds[comp].mean()) if area else 0.0
        if seed_ratio < 0.04:
            component_rejections["low_necrotic_seed_support"] += 1
            continue
        edge_touch = float(np.logical_and(comp, leaf_edge_dil).sum() / max(1, area))
        if edge_touch > 0.82 and area / leaf_area > 0.02 and seed_ratio < 0.16:
            component_rejections["mostly_leaf_edge"] += 1
            continue
        clean[comp] = 255

    clean = cv2.morphologyEx(clean, cv2.MORPH_OPEN, k3, iterations=1)
    area_leaf = float((clean > 0).sum() / max(1, leaf_area))
    comp_count = cv2.connectedComponents((clean > 0).astype(np.uint8), 8)[0] - 1
    class_max = disease_area_max_for_class(source_class)
    reasons = []
    if area_leaf == 0:
        reasons.append("empty_disease_mask")
    if 0 < area_leaf < 0.001:
        reasons.append("tiny_disease_mask")
    if area_leaf > class_max:
        reasons.append("large_disease_mask")
    if comp_count > 50:
        reasons.append("many_components")
    if component_rejections:
        reasons.extend([f"component_reject_{k}:{v}" for k, v in sorted(component_rejections.items())])

    severe = [r for r in reasons if r in {"empty_disease_mask", "tiny_disease_mask", "large_disease_mask", "many_components"}]
    return clean, {
        "accepted": len(severe) == 0,
        "weak_reasons": ";".join(reasons),
        "disease_area_ratio_leaf": area_leaf,
        "component_count": int(comp_count),
        "class_area_max": class_max,
    }

def label_lines_to_masks(lines, shape):
    h, w = shape
    masks = {0: np.zeros((h, w), dtype=np.uint8), 1: np.zeros((h, w), dtype=np.uint8)}
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        cls = int(parts[0])
        vals = [float(v) for v in parts[1:]]
        pts = [[int(round(x * w)), int(round(y * h))] for x, y in zip(vals[0::2], vals[1::2])]
        if cls in masks and len(pts) >= 3:
            cv2.fillPoly(masks[cls], [np.array(pts, dtype=np.int32)], 255)
    return masks

def assert_multiclass_labels(label_root, manifest=None):
    bad = []
    for p in Path(label_root).glob("**/*.txt"):
        text = p.read_text(encoding="utf-8").splitlines()
        classes = []
        for line in text:
            if not line.strip():
                continue
            cls = line.split()[0]
            if cls not in {"0", "1"}:
                bad.append((str(p), "bad_class", cls))
            classes.append(cls)
        if "0" not in classes:
            bad.append((str(p), "missing_leaf_class", ""))
    if manifest is not None and len(manifest):
        for row in manifest.itertuples(index=False):
            if bool(row.is_healthy) and int(row.disease_polygon_count) != 0:
                bad.append((str(row.label_path), "healthy_has_disease_label", ""))
            if (not bool(row.is_healthy)) and str(row.sample_type) == "leaf_disease" and int(row.disease_polygon_count) == 0:
                bad.append((str(row.label_path), "positive_missing_disease_label", ""))
    if bad:
        raise AssertionError(f"Multiclass label validation failed: {bad[:5]}")

def class_result_masks(result, shape):
    h, w = shape
    out = {0: np.zeros((h, w), dtype=np.uint8), 1: np.zeros((h, w), dtype=np.uint8)}
    masks = getattr(result, "masks", None)
    boxes = getattr(result, "boxes", None)
    if masks is None:
        return out
    data = getattr(masks, "data", None)
    if data is None or len(data) == 0:
        return out
    if boxes is not None and getattr(boxes, "cls", None) is not None:
        cls_vals = boxes.cls.detach().cpu().numpy().astype(int).tolist()
    else:
        cls_vals = [0] * len(data)
    for arr, cls_id in zip(data.detach().cpu().numpy(), cls_vals):
        if cls_id not in out:
            continue
        m = (arr > 0.5).astype(np.uint8) * 255
        if m.shape != (h, w):
            m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
        out[cls_id] = cv2.bitwise_or(out[cls_id], m)
    return out

def overlay_multiclass(rgb, leaf_mask, disease_mask, alpha=0.45):
    out = rgb.copy()
    if leaf_mask is not None:
        out = overlay_mask(out, leaf_mask, (40, 200, 90), alpha=alpha)
    if disease_mask is not None:
        out = overlay_mask(out, disease_mask, (80, 140, 255), alpha=0.55)
    return out

def metrics_from_results_multiclass(metrics):
    values = metrics_from_results(metrics)
    names = getattr(metrics, "names", None)
    if names is not None:
        values["names"] = {str(k): str(v) for k, v in dict(names).items()}
    for attr in ["box", "seg"]:
        obj = getattr(metrics, attr, None)
        maps = getattr(obj, "maps", None) if obj is not None else None
        if maps is not None:
            arr = np.array(maps).astype(float).ravel().tolist()
            for i, val in enumerate(arr):
                key = str(names.get(i, i)) if names is not None and hasattr(names, "get") else str(i)
                values[f"pseudo_{attr}_map_class_{key}"] = float(val)
    return values


## Multiclass Pseudo-Label QA
Generate leaf and disease pseudo-labels, keep healthy leaf-only samples, reject weak disease positives, and gate training.

In [ ]:
progress["stage"] = "multiclass_pseudo_label_qa"; save_progress()
rows, accepted_rows, rejected_rows = [], [], []
preview_counts = Counter()

for idx, row in enumerate(source_df.itertuples(index=False), start=1):
    p = Path(row.path)
    rgb = read_rgb(p)
    leaf_mask, before_fill_area, after_fill_area = leaf_tissue_shape_mask(rgb)
    leaf_q = mask_quality(leaf_mask, before_fill_area, after_fill_area)
    bbox = bbox_from_mask(leaf_mask, pad_ratio=0.10)
    crop_rgb, crop_leaf = crop_rgb_mask(rgb, leaf_mask, bbox)
    leaf_lines = masks_to_yolo_lines(crop_leaf, class_id=0, min_area_ratio=0.003, max_contours=4) if leaf_q["accepted"] else []
    is_healthy = bool(row.is_healthy)
    policy = "healthy_leaf_only" if is_healthy else disease_policy(row.source_class)
    disease_mask = np.zeros(crop_leaf.shape, dtype=np.uint8)
    disease_stats = {
        "accepted": False,
        "weak_reasons": "",
        "disease_area_ratio_leaf": 0.0,
        "component_count": 0,
        "class_area_max": disease_area_max_for_class(row.source_class),
    }
    disease_lines = []
    reasons = []
    accepted = False
    sample_type = "rejected"

    if not leaf_q["accepted"] or not leaf_lines:
        reasons.append(("leaf_mask_rejected;" + leaf_q.get("weak_reasons", "")).strip(";"))
    elif is_healthy:
        accepted = True
        sample_type = "healthy_leaf_only"
    elif policy == "focal_positive":
        disease_mask, disease_stats = disease_mask_conservative(crop_rgb, crop_leaf, row.source_class)
        disease_lines = masks_to_yolo_lines(disease_mask, class_id=1, min_area_ratio=0.0004, max_contours=50) if disease_stats["accepted"] else []
        if disease_stats["accepted"] and disease_lines:
            accepted = True
            sample_type = "leaf_disease"
        else:
            reasons.append(("disease_rejected;" + disease_stats.get("weak_reasons", "")).strip(";"))
    else:
        reasons.append(policy)

    rec = {
        "source_class": row.source_class,
        "filename": row.filename,
        "normalized_id": row.normalized_id,
        "is_healthy": is_healthy,
        "source_path": str(p),
        "policy": policy,
        "sample_type": sample_type,
        "accepted": accepted,
        "bbox": ",".join(str(int(v)) for v in bbox),
        "crop_h": int(crop_rgb.shape[0]),
        "crop_w": int(crop_rgb.shape[1]),
        "leaf_polygon_count": len(leaf_lines),
        "disease_polygon_count": len(disease_lines),
        "leaf_mask_area_ratio_crop": float((crop_leaf > 0).mean()),
        "leaf_quality_reasons": leaf_q.get("weak_reasons", ""),
        "disease_area_ratio_leaf": float(disease_stats["disease_area_ratio_leaf"]),
        "disease_component_count": int(disease_stats["component_count"]),
        "disease_class_area_max": float(disease_stats["class_area_max"]),
        "disease_weak_reasons": disease_stats.get("weak_reasons", ""),
        "rejection_reasons": ";".join([x for x in reasons if x]),
    }
    rows.append(rec)
    if accepted:
        accepted_rows.append(rec)
        bucket = "healthy_leaf_only" if is_healthy else "accepted"
    else:
        rejected_rows.append(rec)
        bucket = "rejected"

    if preview_counts[bucket] < 140:
        if bucket == "healthy_leaf_only":
            panel = side_by_side([crop_rgb, overlay_mask(crop_rgb, crop_leaf, (40, 200, 90))])
        elif bucket == "accepted":
            panel = side_by_side([
                crop_rgb,
                overlay_mask(crop_rgb, crop_leaf, (40, 200, 90)),
                overlay_mask(crop_rgb, disease_mask, (80, 140, 255)),
                overlay_multiclass(crop_rgb, crop_leaf, disease_mask),
            ])
        else:
            panel = side_by_side([
                crop_rgb,
                overlay_mask(crop_rgb, crop_leaf, (255, 160, 30)),
                overlay_mask(crop_rgb, disease_mask, (255, 80, 80)),
            ])
        write_rgb(PREVIEW_DIR / bucket / f"{idx:06d}__{row.source_class}__{p.stem}.jpg", panel)
        preview_counts[bucket] += 1

    if idx % 500 == 0:
        pd.DataFrame(rows).to_csv(QUANT_DIR / "multiclass_pseudo_label_inventory_partial.csv", index=False)
        progress["multiclass_qa_processed"] = idx
        save_progress()
        print({"processed": idx, "accepted": len(accepted_rows), "rejected": len(rejected_rows)})

inventory_df = pd.DataFrame(rows)
accepted_df = pd.DataFrame(accepted_rows)
rejections_df = pd.DataFrame(rejected_rows)
inventory_path = QUANT_DIR / "multiclass_pseudo_label_inventory.csv"
rejections_path = QUANT_DIR / "multiclass_pseudo_label_rejections.csv"
qa_path = QUANT_DIR / "multiclass_qa_summary.json"
inventory_df.to_csv(inventory_path, index=False)
rejections_df.to_csv(rejections_path, index=False)

positive_count = int(((accepted_df.get("sample_type", pd.Series(dtype=str)) == "leaf_disease")).sum()) if len(accepted_df) else 0
healthy_count = int(((accepted_df.get("sample_type", pd.Series(dtype=str)) == "healthy_leaf_only")).sum()) if len(accepted_df) else 0
qa_summary = {
    "total_candidates": int(len(inventory_df)),
    "accepted": int(len(accepted_df)),
    "rejected": int(len(rejections_df)),
    "leaf_disease_positive_samples": positive_count,
    "healthy_leaf_only_samples": healthy_count,
    "min_accepted_disease_positives": MIN_ACCEPTED_DISEASE_POSITIVES,
    "min_healthy_negatives": MIN_HEALTHY_NEGATIVES,
    "accepted_by_type": {str(k): int(v) for k, v in accepted_df.get("sample_type", pd.Series(dtype=str)).value_counts().to_dict().items()},
    "rejection_reasons": {k: int(v) for k, v in Counter(";".join(rejections_df.get("rejection_reasons", pd.Series(dtype=str)).fillna("").astype(str)).split(";")).items() if k},
}
qa_path.write_text(json.dumps(qa_summary, indent=2), encoding="utf-8")
progress["multiclass_qa_summary"] = qa_summary
progress["artifacts"].extend([str(inventory_path), str(rejections_path), str(qa_path), str(PREVIEW_DIR)])
if positive_count < MIN_ACCEPTED_DISEASE_POSITIVES or healthy_count < MIN_HEALTHY_NEGATIVES:
    progress["status"] = "failed_qa"
    progress["stage"] = "multiclass_qa_failed_no_training"
    progress["qa_failure_reason"] = "not enough accepted disease positives or healthy negatives"
else:
    progress["stage"] = "multiclass_qa_passed"
save_progress()
print(qa_summary)


## Prepare Multiclass YOLO Dataset
Use accepted samples only, split 70/15/15, and write YOLO-seg labels with class 0 and class 1.

In [ ]:
if progress.get("status") != "failed_qa":
    progress["stage"] = "prepare_multiclass_yolo_dataset"; save_progress()
    accepted_df = split_stratified(accepted_df, "source_class", 0.70, 0.15)
    accepted_df["row_id"] = [f"multi9_{i:06d}" for i in range(len(accepted_df))]
    if set(accepted_df["normalized_id"]) & holdout_norm_ids:
        raise AssertionError("Holdout leaked into v9 train/val/test")
    for split in ["train", "val", "test"]:
        (PREPARED_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (PREPARED_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

    prep_rows = []
    for idx, row in enumerate(accepted_df.itertuples(index=False), start=1):
        p = Path(row.source_path)
        rgb = read_rgb(p)
        leaf_mask, _, _ = leaf_tissue_shape_mask(rgb)
        bbox = bbox_from_mask(leaf_mask, pad_ratio=0.10)
        crop_rgb, crop_leaf = crop_rgb_mask(rgb, leaf_mask, bbox)
        leaf_lines = masks_to_yolo_lines(crop_leaf, class_id=0, min_area_ratio=0.003, max_contours=4)
        disease_mask = np.zeros(crop_leaf.shape, dtype=np.uint8)
        disease_lines = []
        if not bool(row.is_healthy) and str(row.sample_type) == "leaf_disease":
            disease_mask, disease_stats = disease_mask_conservative(crop_rgb, crop_leaf, row.source_class)
            disease_lines = masks_to_yolo_lines(disease_mask, class_id=1, min_area_ratio=0.0004, max_contours=50)
        lines = leaf_lines + disease_lines
        out_name = f"{row.row_id}__{row.source_class}__{p.stem}.jpg"
        image_out = PREPARED_DIR / "images" / row.split / out_name
        label_out = PREPARED_DIR / "labels" / row.split / (Path(out_name).stem + ".txt")
        write_rgb(image_out, crop_rgb)
        label_out.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
        prep_rows.append({
            **row._asdict(),
            "image_path": str(image_out),
            "label_path": str(label_out),
            "bbox_runtime": ",".join(str(int(v)) for v in bbox),
            "leaf_polygon_count_runtime": len(leaf_lines),
            "disease_polygon_count_runtime": len(disease_lines),
            "has_leaf_label": len(leaf_lines) > 0,
            "has_disease_label": len(disease_lines) > 0,
        })
        if idx % 1000 == 0:
            print({"prepared": idx, "total": len(accepted_df)})

    prep_df = pd.DataFrame(prep_rows)
    split_manifest = QUANT_DIR / "multiclass_training_split_manifest.csv"
    prep_df.to_csv(split_manifest, index=False)
    data_yaml = PREPARED_DIR / "data.yaml"
    data_yaml.write_text(f"path: {PREPARED_DIR}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: leaf\n  1: disease_region\n", encoding="utf-8")
    assert_multiclass_labels(PREPARED_DIR / "labels", prep_df)
    progress["split_counts"] = {k: int(v) for k, v in accepted_df["split"].value_counts().to_dict().items()}
    progress["prepared_counts"] = {
        "total": int(len(prep_df)),
        "leaf_only": int((prep_df["sample_type"] == "healthy_leaf_only").sum()),
        "leaf_disease": int((prep_df["sample_type"] == "leaf_disease").sum()),
    }
    progress["artifacts"].extend([str(split_manifest), str(data_yaml)])
    save_progress()
    print(progress["split_counts"])


## Train And Quantitative Pseudo Metrics
Train YOLO multiclass and save separate train/test pseudo metrics.

In [ ]:
if progress.get("status") != "failed_qa":
    progress["stage"] = "train_multiclass_yolo"; save_progress()
    model = YOLO("yolo11n-seg.pt")
    train_result = model.train(
        data=str(PREPARED_DIR / "data.yaml"),
        epochs=25,
        imgsz=640,
        batch=16,
        workers=2,
        seed=SEED,
        device=DEVICE,
        project=str(RUNS_DIR),
        name="yolo11n_multiclass_v9",
        exist_ok=True,
        patience=8,
        verbose=False,
    )
    train_dir = Path(getattr(train_result, "save_dir", RUNS_DIR / "yolo11n_multiclass_v9"))
    best_model = WORKING_DIR / "best_multiclass_model.pt"
    if (train_dir / "weights" / "best.pt").exists():
        shutil.copy2(train_dir / "weights" / "best.pt", best_model)
    trained = YOLO(str(best_model if best_model.exists() else train_dir / "weights" / "last.pt"))
    metric_outputs = {}
    for split in ["train", "test"]:
        m = trained.val(
            data=str(PREPARED_DIR / "data.yaml"),
            split=split,
            imgsz=640,
            batch=16,
            device=DEVICE,
            project=str(RUNS_DIR),
            name=f"eval_{split}",
            exist_ok=True,
            verbose=False,
        )
        vals = metrics_from_results_multiclass(m)
        (QUANT_DIR / f"{split}_pseudo_metrics.json").write_text(json.dumps(vals, indent=2), encoding="utf-8")
        pd.DataFrame([vals]).to_csv(QUANT_DIR / f"{split}_pseudo_metrics.csv", index=False)
        metric_outputs[split] = vals
    progress["train_test_pseudo_metrics"] = metric_outputs
    progress["artifacts"].extend([str(best_model), str(train_dir)])
    save_progress()
    print(metric_outputs)


## Healthy False Positive Metrics
Measure disease false positives on healthy test crops.

In [ ]:
if progress.get("status") != "failed_qa":
    progress["stage"] = "healthy_false_positive_metrics"; save_progress()
    trained = YOLO(str(WORKING_DIR / "best_multiclass_model.pt"))
    healthy_test = prep_df[(prep_df["split"] == "test") & (prep_df["sample_type"] == "healthy_leaf_only")]
    rows = []
    for row in healthy_test.itertuples(index=False):
        rgb = read_rgb(row.image_path)
        pred = trained.predict(source=rgb, imgsz=640, device=DEVICE, conf=0.25, verbose=False)[0]
        masks = class_result_masks(pred, rgb.shape[:2])
        disease_area = float((masks[1] > 0).mean())
        rows.append({
            "row_id": row.row_id,
            "source_class": row.source_class,
            "image_path": row.image_path,
            "pred_disease_area_ratio": disease_area,
            "has_false_positive": disease_area > 0.001,
        })
    df = pd.DataFrame(rows)
    metrics = {
        "healthy_test_count": int(len(df)),
        "healthy_false_positive_rate_test": float(df["has_false_positive"].mean()) if len(df) else None,
        "mean_pred_disease_area_healthy_test": float(df["pred_disease_area_ratio"].mean()) if len(df) else None,
        "empty_prediction_rate_healthy_test": float((df["pred_disease_area_ratio"] <= 0.001).mean()) if len(df) else None,
    }
    (QUANT_DIR / "healthy_false_positive_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    df.to_csv(QUANT_DIR / "healthy_false_positive_metrics.csv", index=False)
    progress["healthy_false_positive_metrics"] = metrics
    save_progress()
    print(metrics)


## Real Unseen Holdout Predictions
Save original, YOLO leaf prediction, YOLO disease prediction, and combined prediction for all 67 holdout images.

In [ ]:
progress["stage"] = "real_holdout_predictions"; save_progress()
trained = YOLO(str(WORKING_DIR / "best_multiclass_model.pt")) if progress.get("status") != "failed_qa" and (WORKING_DIR / "best_multiclass_model.pt").exists() else None
holdout_rows = []
diag_rows = []

for rec in holdout_df.itertuples(index=False):
    p = Path(rec.path)
    rgb = read_rgb(p)
    leaf_mask, _, _ = leaf_tissue_shape_mask(rgb)
    bbox = bbox_from_mask(leaf_mask, pad_ratio=0.10)
    crop_rgb, crop_leaf = crop_rgb_mask(rgb, leaf_mask, bbox)

    if trained is not None:
        pred = trained.predict(source=crop_rgb, imgsz=640, device=DEVICE, conf=0.25, verbose=False)[0]
        crop_masks = class_result_masks(pred, crop_rgb.shape[:2])
    else:
        crop_masks = {0: np.zeros(crop_leaf.shape, dtype=np.uint8), 1: np.zeros(crop_leaf.shape, dtype=np.uint8)}
    full_leaf_pred = paste_crop_mask(crop_masks[0], rgb.shape[:2], bbox)
    full_disease_pred = paste_crop_mask(crop_masks[1], rgb.shape[:2], bbox)

    panel = side_by_side([
        rgb,
        overlay_mask(rgb, full_leaf_pred, (40, 200, 90)),
        overlay_mask(rgb, full_disease_pred, (80, 140, 255)),
        overlay_multiclass(rgb, full_leaf_pred, full_disease_pred),
    ])
    out_path = REAL_HOLDOUT_DIR / "by_category" / rec.category / f"{Path(rec.holdout_filename).stem}_orig_leafpred_diseasepred_combined.jpg"
    write_rgb(out_path, panel)
    holdout_rows.append({
        "category": rec.category,
        "holdout_filename": rec.holdout_filename,
        "original_filename": rec.original_filename,
        "bbox_for_inference": ",".join(str(int(v)) for v in bbox),
        "pred_leaf_area_ratio": float((full_leaf_pred > 0).mean()),
        "pred_disease_area_ratio": float((full_disease_pred > 0).mean()),
        "qualitative_path": str(out_path),
        "note": "real unseen holdout prediction; no pseudo metric",
    })

    is_holdout_healthy = (rec.category == "healthy_leaves") or ("healthy" in rec.holdout_filename.lower())
    diag_disease = np.zeros(crop_leaf.shape, dtype=np.uint8)
    if not is_holdout_healthy and disease_policy(rec.original_filename) == "focal_positive":
        diag_disease, _ = disease_mask_conservative(crop_rgb, crop_leaf, rec.original_filename)
    full_diag_disease = paste_crop_mask(diag_disease, rgb.shape[:2], bbox)
    diag_panel = side_by_side([
        rgb,
        overlay_mask(rgb, leaf_mask, (40, 200, 90)),
        overlay_mask(rgb, full_diag_disease, (255, 180, 40)),
    ])
    diag_path = HOLDOUT_DIAG_DIR / "by_category" / rec.category / f"{Path(rec.holdout_filename).stem}_diagnostic_leaf_diseaseheuristic.jpg"
    write_rgb(diag_path, diag_panel)
    diag_rows.append({
        "category": rec.category,
        "holdout_filename": rec.holdout_filename,
        "diagnostic_path": str(diag_path),
        "note": "diagnostic heuristic reference only; not training, not ground truth, not metric",
    })

holdout_csv = WORKING_DIR / "real_holdout_predictions.csv"
diag_csv = WORKING_DIR / "holdout_diagnostic_reference.csv"
pd.DataFrame(holdout_rows).to_csv(holdout_csv, index=False)
pd.DataFrame(diag_rows).to_csv(diag_csv, index=False)
progress["real_holdout_predictions"] = {"count": int(len(holdout_rows)), "note": "holdout excluded from training and used only for qualitative unseen predictions"}
progress["artifacts"].extend([str(holdout_csv), str(diag_csv)])
save_progress()
print(progress["real_holdout_predictions"])


## Qualitative Test Results
Save pseudo-vs-pred panels on a quantitative test sample.

In [ ]:
if progress.get("status") != "failed_qa":
    progress["stage"] = "qualitative_multiclass_test"; save_progress()
    trained = YOLO(str(WORKING_DIR / "best_multiclass_model.pt"))
    sample = prep_df[prep_df["split"] == "test"].sample(n=min(120, int((prep_df["split"] == "test").sum())), random_state=SEED)
    for row in sample.itertuples(index=False):
        rgb = read_rgb(row.image_path)
        label_masks = label_file_to_mask(row.label_path, rgb.shape[:2])
        pseudo_leaf = label_masks if isinstance(label_masks, np.ndarray) else label_masks
        masks_from_label = label_lines_to_masks(Path(row.label_path).read_text(encoding="utf-8").splitlines(), rgb.shape[:2])
        pred = trained.predict(source=rgb, imgsz=640, device=DEVICE, conf=0.25, verbose=False)[0]
        pred_masks = class_result_masks(pred, rgb.shape[:2])
        panel = side_by_side([
            rgb,
            overlay_multiclass(rgb, masks_from_label[0], masks_from_label[1]),
            overlay_multiclass(rgb, pred_masks[0], pred_masks[1]),
        ])
        write_rgb(QUAL_TEST_DIR / f"{row.row_id}_pseudo_pred.jpg", panel)
    progress["artifacts"].append(str(QUAL_TEST_DIR))
    save_progress()


## Finalize
Zip deliverables and write final run summary.

In [ ]:
progress["stage"] = "finalize"; save_progress()
for folder, zip_name in [
    (PREPARED_DIR, "prepared_multiclass_yolo_dataset.zip"),
    (PREVIEW_DIR, "multiclass_pseudo_labels_preview.zip"),
    (REAL_HOLDOUT_DIR, "real_holdout_predictions.zip"),
    (HOLDOUT_DIAG_DIR, "holdout_diagnostic_reference.zip"),
    (QUAL_TEST_DIR, "qualitative_multiclass_results.zip"),
    (QUANT_DIR, "quantitative_metrics.zip"),
]:
    if folder.exists():
        z = zip_dir(folder, WORKING_DIR / zip_name)
        progress["artifacts"].append(str(z))
if progress.get("status") != "failed_qa":
    progress["status"] = "complete"
for bulky in [PREPARED_DIR, PREVIEW_DIR, REAL_HOLDOUT_DIR, HOLDOUT_DIAG_DIR, QUAL_TEST_DIR, RUNS_DIR]:
    if bulky.exists():
        shutil.rmtree(bulky)
progress["stage"] = "done"
progress["artifacts"] = sorted(set(progress["artifacts"]))
save_progress()
print(json.dumps({"status": progress["status"], "stage": progress["stage"], "artifact_count": len(progress["artifacts"])}, indent=2))
